# 04 — Predictive Analysis

Use the course dataset's documented binary landing-success target. Verify the target mapping and exclude post-outcome leakage before fitting.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
path = Path('../data/cleaned_launches.csv')
if not path.exists():
    raise FileNotFoundError('Create data/cleaned_launches.csv from your course dataset first.')
df = pd.read_csv(path)
print('Shape:', df.shape)
print(df.columns.tolist())
df.head()

## Define target and predictors

Update these names to match the dataset. The target must be binary and must not be included in the predictors.

In [ ]:
TARGET = 'landing_success'  # confirm mapping from the course dataset
if TARGET not in df.columns:
    raise KeyError(f'Missing target column: {TARGET}')
df = df[df[TARGET].isin([0,1])].copy()
leakage_cols = [TARGET, 'landing_outcome', 'landing_date', 'reused']
X = df.drop(columns=[c for c in leakage_cols if c in df.columns])
y = df[TARGET].astype(int)
# Drop identifier/text fields unsuitable as direct predictors; adapt this list to your dataset.
X = X.drop(columns=[c for c in ['id', 'launch_id', 'details', 'links'] if c in X.columns])
cat_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()
print('Target balance:'); print(y.value_counts(dropna=False))
print('Categorical:', cat_cols); print('Numeric:', num_cols)

## Split, preprocess, fit, evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
pre = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])
model = Pipeline([('preprocess', pre), ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced'))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Test rows:', len(y_test))
print('Accuracy:', round(accuracy_score(y_test, pred), 4))
print('Precision:', round(precision_score(y_test, pred, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, pred, zero_division=0), 4))
print('F1:', round(f1_score(y_test, pred, zero_division=0), 4))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

## Interpretation

Record the metrics from the output above. Explain the confusion matrix, class balance, limitations, and why the test set was kept separate from tuning. Do not copy generic or guessed metrics into the report.